# Screening quantity on document images

**Computes:** Ceiling, predicted and realized gain, and the bound on document images.

**Inputs:** DIBCO images (downloaded by the notebook).  
**Outputs:** `results/04/results_04.json` and a text report.


In [ ]:
import sys, os, platform, json, time, pickle
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.abspath(os.path.join(RESULTS_DIR, "04"))
os.makedirs(OUTDIR, exist_ok=True)
import numpy as np, scipy
print(f"colab {IN_COLAB} | python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__}")
print(f"output dir: {OUTDIR}")

In [ ]:
import subprocess, glob
REPO = "https://github.com/RichSu95/Document_Binarization_Collection.git"
ROOT = os.environ.get("DIBCO_ROOT") or ("/content/dibco_all" if "google.colab" in sys.modules else os.path.abspath(os.path.join(OUTDIR, "dibco_all")))
DATA_DIR = os.path.join(ROOT, "repo", "SauvolaNet", "Dataset")
if not os.path.isdir(DATA_DIR):
    os.makedirs(ROOT, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "-q", REPO, os.path.join(ROOT, "repo")], check=True)
n_src = len(glob.glob(os.path.join(DATA_DIR, "*DIBCO*_source.*")))
print(f"{n_src} DIBCO source images")
assert n_src >= 90, "expected at least 90 DIBCO images"

In [ ]:
import glob, os, json, time
import numpy as np
from PIL import Image
from scipy.ndimage import sobel, uniform_filter, binary_dilation, binary_erosion
from scipy import stats

NB = 12
BINS = 8
PER_IMG = 10_000
M_T1, M1, M2 = 2.0, 1.1, 10.1
A_REP = 0.15
WIN = 25
K_SAUV, R_SAUV = 0.2, 0.5
PRECOND = 0.077
EPS = 1e-9

def _disc(v, nb=NB):
    q = np.quantile(v, np.linspace(0, 1, nb + 1)[1:-1])
    return np.searchsorted(q, v)

def _H(*cols):
    idx = np.zeros(len(cols[0]), dtype=np.int64)
    for c in cols:
        idx = idx * (c.max() + 1) + c
    cnt = np.bincount(idx)
    p = cnt[cnt > 0] / cnt.sum()
    return float(-(p * np.log(p)).sum())

def mi(y, x):
    return _H(y) + _H(x) - _H(y, x)

def cmi(y, s, u):
    return _H(y, u) + _H(s, u) - _H(y, s, u) - _H(u)

def cmi_with_null(y, s, u, n_perm=200, seed=0):
    obs = cmi(y, s, u)
    r = np.random.default_rng(seed)
    null = []
    order = np.argsort(u, kind="stable")
    bounds = np.searchsorted(u[order], np.arange(u.max() + 2))
    for _ in range(n_perm):
        sp = s.copy()
        for a, b in zip(bounds[:-1], bounds[1:]):
            if b - a > 1:
                blk = order[a:b]
                sp[blk] = s[r.permutation(blk)]
        null.append(cmi(y, sp, u))
    null = np.array(null)
    return obs, float(np.median(null)), float((null >= obs).mean()), obs - float(np.median(null))

def excess(y, x, n_perm=40, seed=0):
    rng = np.random.default_rng(seed)
    yb = y.astype(np.int64)
    X = _disc(x)
    obs = mi(yb, X)
    nul = float(np.median([mi(rng.permutation(yb), X) for _ in range(n_perm)]))
    return obs - nul

def fit_h(u_tr, s_tr, y_tr, u_te, s_te, bins=BINS):
    qu = np.quantile(u_tr, np.linspace(0, 1, bins + 1)); qu[0] -= 1e-9
    qs = np.quantile(s_tr, np.linspace(0, 1, bins + 1)); qs[0] -= 1e-9
    iu = np.clip(np.searchsorted(qu, u_tr) - 1, 0, bins - 1)
    isx = np.clip(np.searchsorted(qs, s_tr) - 1, 0, bins - 1)
    tot = np.zeros((bins, bins)); cnt = np.zeros((bins, bins))
    np.add.at(tot, (iu, isx), y_tr); np.add.at(cnt, (iu, isx), 1.0)
    grid = np.where(cnt > 0, tot / np.maximum(cnt, 1), y_tr.mean())
    return grid[np.clip(np.searchsorted(qu, u_te) - 1, 0, bins - 1),
                np.clip(np.searchsorted(qs, s_te) - 1, 0, bins - 1)]

def fcm_u(I, v1, v2, m):
    d1 = np.abs(I - v1) + EPS
    d2 = np.abs(I - v2) + EPS
    return 1.0 / (1.0 + np.clip((d1 / d2) ** (2.0 / (m - 1.0)), 0, 1e12))

def fit_protos(I, m=M_T1, iters=30):
    v1, v2 = np.percentile(I, 5), np.percentile(I, 95)
    for _ in range(iters):
        u = fcm_u(I, v1, v2, m)
        w1, w2 = u ** m, (1 - u) ** m
        v1n = (w1 * I).sum() / max(w1.sum(), EPS)
        v2n = (w2 * I).sum() / max(w2.sum(), EPS)
        done = abs(v1n - v1) < 1e-6 and abs(v2n - v2) < 1e-6
        v1, v2 = v1n, v2n
        if done:
            break
    return float(v1), float(v2)

def src_gradient(I):
    return uniform_filter(np.hypot(sobel(I, axis=0), sobel(I, axis=1)), size=5)

def src_sauvola(I, win=WIN):
    mu = uniform_filter(I, win)
    sd = np.sqrt(np.maximum(uniform_filter(I * I, win) - mu * mu, 0.0))
    T = mu * (1.0 + K_SAUV * (sd / R_SAUV - 1.0))
    return I - T

def dibco_pairs(data_dir):
    P = sorted((f, f.replace("_source.", "_target."))
               for f in glob.glob(os.path.join(data_dir, "*DIBCO*_source.*")))
    return [(a, b) for a, b in P if os.path.exists(b)]

def load_image(f, g, idx, extra_windows=()):
    I = np.array(Image.open(f).convert("L"), dtype=float) / 255.0
    G = np.array(Image.open(g).convert("L")) < 128
    v1, v2 = fit_protos(I)
    u = fcm_u(I, v1, v2, M_T1)
    dec = u > 0.5
    y_err = dec != G
    y_edge = binary_dilation(G) ^ binary_erosion(G)
    a, b = fcm_u(I, v1, v2, M1), fcm_u(I, v1, v2, M2)
    w_pair = np.abs(a - b)
    uc = np.clip(u, 0, 1)
    w_power = uc ** (1 - A_REP) - uc ** (1 + A_REP)
    feats = dict(u=u, grad=src_gradient(I), sauvola=src_sauvola(I),
                 w_pair=w_pair, w_power=w_power)
    for wv in extra_windows:
        feats[f"sauvola_w{wv}"] = src_sauvola(I, wv)
    rng = np.random.default_rng(1000 + idx)
    n = I.size
    k = rng.choice(n, min(PER_IMG, n), replace=False)
    out = {key: v.ravel()[k].astype(np.float64) for key, v in feats.items()}
    out["err"] = y_err.ravel()[k]; out["edge"] = y_edge.ravel()[k]
    key = os.path.splitext(os.path.basename(f).replace("TRAIN_", "").replace("_source", ""))[0]
    meta = dict(name=key, year=key.split("_")[0].replace("DIBCO", ""), shape=list(I.shape),
                err_rate=float(y_err.mean()), edge_rate=float(y_edge.mean()),
                f_t1=float(2 * (dec & G).sum() / max(dec.sum() + G.sum(), 1)))
    return out, meta

def split_folds(metas, seed=7):
    rng = np.random.default_rng(seed)
    fold = np.zeros(len(metas), dtype=int)
    for y in sorted({m["year"] for m in metas}):
        ix = np.array([i for i, m in enumerate(metas) if m["year"] == y])
        ix = ix[rng.permutation(len(ix))]
        fold[ix[1::2]] = 1
    return fold

def pool(samples, ids, keys):
    return {k: np.concatenate([samples[i][k] for i in ids]) for k in keys}

def analyse_fold(samples, tr_ids, te_ids, target, source, n_perm_cmi=200, n_boot=200, seed=0):
    keys = ["u", source, "w_pair", "w_power", target]
    TR, TE = pool(samples, tr_ids, keys), pool(samples, te_ids, keys)
    y_tr, y_te = TR[target].astype(float), TE[target]
    ceil_tr = excess(TR[target], TR["u"], seed=1)
    obs, nul, p, cmi_exc = cmi_with_null(TR[target].astype(np.int64), _disc(TR[source]), _disc(TR["u"]),
                                         n_perm=n_perm_cmi, seed=2)
    eta = cmi_exc / max(ceil_tr, 1e-12)
    h_us = fit_h(TR["u"], TR[source], y_tr, TE["u"], TE[source])
    h_u = fit_h(TR["u"], np.zeros_like(TR["u"]), y_tr, TE["u"], np.zeros_like(TE["u"]))
    I = {nm: excess(y_te, x, seed=10 + j) for j, (nm, x) in enumerate(
        (("u", TE["u"]), ("s", TE[source]), ("h_u", h_u), ("h_us", h_us),
         ("w_pair", TE["w_pair"]), ("w_power", TE["w_power"])))}
    ceil = I["u"]
    realized = I["h_us"] / max(ceil, 1e-12) - 1.0
    rng = np.random.default_rng(100 + seed)
    g_s, g_u, g_b = [], [], []
    for b in range(n_boot):
        pick = rng.choice(te_ids, len(te_ids), replace=True)
        B = pool(samples, pick, keys)
        hb = fit_h(TR["u"], TR[source], y_tr, B["u"], B[source])
        e_h = excess(B[target], hb, n_perm=10, seed=b)
        e_s = excess(B[target], B[source], n_perm=10, seed=b)
        e_u = excess(B[target], B["u"], n_perm=10, seed=b)
        g_s.append(e_h - e_s); g_u.append(e_h - e_u); g_b.append(e_h - max(e_s, e_u))
    ci = lambda v: [float(x) for x in np.percentile(v, [2.5, 97.5])]
    per = []
    for i in te_ids:
        S = samples[i]
        if S[target].sum() < 50 or (~S[target]).sum() < 50:
            continue
        hi = fit_h(TR["u"], TR[source], y_tr, S["u"], S[source])
        e_h = excess(S[target], hi, n_perm=10, seed=i)
        e_s = excess(S[target], S[source], n_perm=10, seed=i)
        e_u = excess(S[target], S["u"], n_perm=10, seed=i)
        per.append(dict(i=int(i), over_s=e_h - e_s, over_u=e_h - e_u, over_best=e_h - max(e_s, e_u)))
    def tests(v):
        v = np.array(v)
        return dict(n=int(len(v)), median=float(np.median(v)), mean=float(v.mean()),
                    t_p=float(stats.ttest_1samp(v, 0.0, alternative="greater").pvalue),
                    w_p=float(stats.wilcoxon(v, alternative="greater").pvalue))
    bins_sweep = {}
    for bn in (4, 6, 8, 12):
        hb = fit_h(TR["u"], TR[source], y_tr, TE["u"], TE[source], bins=bn)
        bins_sweep[bn] = float(excess(y_te, hb, n_perm=15, seed=bn) - I["s"])
    return dict(target=target, source=source, n_train_img=len(tr_ids), n_test_img=len(te_ids),
                n_train_px=int(len(y_tr)), n_test_px=int(len(y_te)), y_rate_test=float(np.mean(y_te)),
                ceiling_train=ceil_tr, cmi_raw=obs, cmi_null=nul, cmi_p=p, cmi_excess=cmi_exc, eta=eta,
                I=I, ceiling=ceil, precondition_met=bool(ceil > PRECOND),
                realized_headroom=realized, tightness=realized / eta if eta > 0 else float("nan"),
                gain_over_s=I["h_us"] - I["s"], gain_over_u=I["h_us"] - I["u"],
                gain_over_best=I["h_us"] - max(I["u"], I["s"]),
                stronger_single="u" if I["u"] >= I["s"] else "s",
                boot_over_s=ci(g_s), boot_over_u=ci(g_u), boot_over_best=ci(g_b),
                perimage_over_s=tests([r["over_s"] for r in per]),
                perimage_over_u=tests([r["over_u"] for r in per]),
                perimage_over_best=tests([r["over_best"] for r in per]),
                bound_ratio_pair=I["w_pair"] / max(ceil, 1e-12),
                bound_ratio_power=I["w_power"] / max(ceil, 1e-12),
                bins_gain_over_s=bins_sweep)

In [ ]:
t0 = time.time()
PAIRS = dibco_pairs(DATA_DIR)
samples, metas = [], []
for i, (f, g) in enumerate(PAIRS):
    s, m = load_image(f, g, i, extra_windows=(15, 51))
    samples.append(s); metas.append(m)
er = np.array([m["err_rate"] for m in metas]); ft = np.array([m["f_t1"] for m in metas])
print(f"{len(samples)} images loaded in {time.time()-t0:.0f}s")
print(f"type-1 F-measure median {np.median(ft):.3f}")
print(f"type-1 error rate median {np.median(er):.4f} [{er.min():.4f}, {er.max():.4f}]")
fold = split_folds(metas)
A, B = np.where(fold == 0)[0], np.where(fold == 1)[0]
print(f"folds: {len(A)} and {len(B)} images")

In [ ]:
t0 = time.time(); MAIN = []
for target in ("err", "edge"):
    for source in ("grad", "sauvola"):
        for fname, (tr, te) in (("A->B", (A, B)), ("B->A", (B, A))):
            r = analyse_fold(samples, tr, te, target, source, n_boot=50, seed=len(MAIN))
            r["fold"] = fname; MAIN.append(r)
            I = r["I"]
            print(f"{target:5s} {source:8s} {fname}  ceiling {r['ceiling']:.4f} (>{PRECOND}: {r['precondition_met']!s:5s}) "
                  f"eta {r['eta']:.3f}  realized {r['realized_headroom']:+.3f}  | I(u) {I['u']:.4f} I(s) {I['s']:.4f} "
                  f"I(h) {I['h_us']:.4f}  | over s {r['gain_over_s']:+.4f}  over best {r['gain_over_best']:+.4f}  [{time.time()-t0:.0f}s]", flush=True)

In [ ]:
ROB = []
for source in ("grad", "sauvola", "sauvola_w15", "sauvola_w51"):
    for fname, (tr, te) in (("A->B", (A, B)), ("B->A", (B, A))):
        keys = ["u", source, "err"]
        TR, TE = pool(samples, tr, keys), pool(samples, te, keys)
        ceil_tr = excess(TR["err"], TR["u"], seed=1)
        _, _, p, ce = cmi_with_null(TR["err"].astype(np.int64), _disc(TR[source]), _disc(TR["u"]), n_perm=100, seed=2)
        ceil = excess(TE["err"], TE["u"], seed=10)
        row = dict(source=source, fold=fname, eta=ce / ceil_tr)
        for bn in (4, 6, 8, 12):
            h = fit_h(TR["u"], TR[source], TR["err"].astype(float), TE["u"], TE[source], bins=bn)
            row[f"realized_b{bn}"] = excess(TE["err"], h, seed=11) / ceil - 1
        ROB.append(row)
        print(f"{source:12s} {fname}: eta {row['eta']:.3f} | realized by grid " +
              "  ".join(f"{bn}x{bn}: {row[f'realized_b{bn}']:+.3f}" for bn in (4, 6, 8, 12)), flush=True)

In [ ]:
# Report.
LINES = []
def say(s=""):
    print(s); LINES.append(s)
def ptxt(t): return f"t p={t['t_p']:.1e}, W p={t['w_p']:.1e}, n={t['n']}, median {t['median']:+.4f}"
def both_pass(rows, key):
    return all(r[key]["t_p"] < 0.05 and r[key]["w_p"] < 0.05 for r in rows)
get = lambda tg, src: [r for r in MAIN if r["target"] == tg and r["source"] == src]

say("=" * 78); say("04 -- screening quantity on document images (DIBCO, 100 images, 8 editions)"); say("=" * 78)
say(f"type-1 F-measure median {np.median(ft):.3f}")
say("")
say("PRECONDITION  held-out I(Y;u) > 0.077 nats")
for tg in ("err", "edge"):
    rr = [r for r in MAIN if r["target"] == tg and r["source"] == "grad"]
    say(f"    {tg:5s}: " + ", ".join(f"{r['fold']} {r['ceiling']:.4f}" for r in rr) +
        f"  -> {'MET on both folds' if all(r['precondition_met'] for r in rr) else 'NOT met'}")
say("")
n_ok = sum(r["realized_headroom"] <= r["eta"] for r in MAIN) + sum(
    row[f"realized_b{bn}"] <= row["eta"] for row in ROB for bn in (4, 6, 8, 12))
n_all = len(MAIN) + 4 * len(ROB)
say(f"BOUND  realized <= eta in {n_ok}/{n_all} configurations")
say("")
for src in ("grad", "sauvola"):
    for r in get("err", src):
        rb = [row for row in ROB if row["source"] == src and row["fold"] == r["fold"]][0]
        say(f"    {src:8s} {r['fold']}: eta {r['eta']:.3f} | realized {r['realized_headroom']:+.3f} at 8x8 "
            f"(tightness {r['tightness']:.2f}) | 12x12 {rb['realized_b12']:+.3f} | 4x4 {rb['realized_b4']:+.3f}")
say("")
for tg in ("err", "edge"):
    for src in ("grad", "sauvola"):
        rr = get(tg, src)
        say(f"    {tg:5s} {src:8s}: {'PASS' if both_pass(rr, 'perimage_over_s') else 'FAIL'}   stronger single source: "
            + "/".join(r["stronger_single"] for r in rr))
        for r in rr: say(f"        {r['fold']}: pooled {r['gain_over_s']:+.4f} CI [{r['boot_over_s'][0]:+.4f}, {r['boot_over_s'][1]:+.4f}] | {ptxt(r['perimage_over_s'])}")
say("")
for tg in ("err", "edge"):
    for src in ("grad", "sauvola"):
        rr = get(tg, src)
        say(f"    {tg:5s} {src:8s}: {'PASS' if both_pass(rr, 'perimage_over_best') else 'FAIL'}")
        for r in rr: say(f"        {r['fold']}: pooled {r['gain_over_best']:+.4f} CI [{r['boot_over_best'][0]:+.4f}, {r['boot_over_best'][1]:+.4f}] | {ptxt(r['perimage_over_best'])}")
say("")
tp = [r["bound_ratio_pair"] for r in MAIN]; tw = [r["bound_ratio_power"] for r in MAIN]
say(f"DIBCO  I(Y;w)/I(Y;u): fuzzifier pair {min(tp):.3f}-{max(tp):.3f}, power construction {min(tw):.3f}-{max(tw):.3f}"
    f"  -> {'all below 1' if max(tp + tw) < 1 else 'SOME ABOVE 1'}")

payload = dict(notebook="04", n_images=len(samples), f_t1_median=float(np.median(ft)),
               fold=fold.tolist(), main=MAIN, robustness=ROB, metas=metas)
jp = os.path.join(OUTDIR, "results_04.json"); tp_ = os.path.join(OUTDIR, "results_04_report.txt")
json.dump(payload, open(jp, "w"), indent=1, default=float)
open(tp_, "w").write("\n".join(LINES) + "\n")
print(f"\nwritten: {jp}\nwritten: {tp_}")